# 9. M0 interpretations: populations of real individuals

Every tutorial so far worked at **M1**, the model level: `part motors :
MotorChoice[4]` *describes* an aircraft as having four motor stations.
**M0** is the level below -- the four actual motors one particular
aircraft *has*, each with its own identity, its own attribute values,
and (for behaviors) its own lifetime. `longeron.m0` builds such
populations directly on the interpreter: an `Interpretation` is a set
of `Individual` runtime instances with stable `qname#index` ids, and
every query -- sequences, roll-ups, sampling -- runs over those actual
individuals instead of the model's descriptions. The working model
throughout is tutorial 7's multi-mission UAV catalog
(`examples/uav_missions.sysml`): real architectures with
physics-derived metrics, whose motors, packs, and spars are worth
holding one at a time.

**You will learn how to:**

- build a nominal interpretation of a real mission architecture and
  address every individual by its stable id (`interpret`,
  `individuals`);
- read features as KerML Annex A sequences (`sequences`);
- roll metrics up over the actual population, and catch the
  hand-encoded population shortcuts the M1 build-ups lean on
  (`rollup`, `Interpretation.gaps`);
- Monte-Carlo the catalog with the seeded random strategy
  (`strategy="random"`, `sample`) and read the mass spread it draws;
- turn a recorded state-machine execution into occurrence individuals
  with lifetimes (`from_timeline`) -- the same representation;
- read a trade-study architecture as a partial interpretation whose
  individuals reproduce the trades metrics (`from_architecture`);
- serialize an interpretation (`to_dict`) -- and know why the standard
  API projection never carries it.

**Prerequisites:** tutorial 3 (instantiation and expressions) and
tutorial 7 (the UAV catalog and its trade studies); tutorial 4's state
machine returns here as a population of occurrences. `longeron.m0`
itself is stdlib-only -- only the trade-study cells use
`longeron.analysis`.

In [ ]:
import longeron
from longeron import m0

model = longeron.load("../examples/uav_missions.sysml")

## A real architecture, as a population with names

Tutorial 7's ISR study ended with a winner: the winged VTOL with
standard motors, slim props, the big pack, the EO/IR ball, and a
carbon spar -- 147 minutes on station. At M1 that mix is six variant
pins on `UavMissions::IsrUav`. `m0.interpret` builds the aircraft
those pins describe: the **nominal** population expands exact
multiplicities fully, ranges take their lower bound (the same choice
`Interpreter.instantiate` makes), and every individual gets a stable
`qname#index` id -- the root is `#0`, nested populations index per
feature, singletons omit the index.

What to look for in the output: four *distinct* motor individuals an
engineer can point at -- `motors#3` is a thing, not a count -- each
carrying its own 92-gram mass; the battery and the sensor are
singletons; 13 individuals in all.

In [ ]:
ISR_MIX = {
    "airframe": "vtolWing",
    "motors": "stdMotor",
    "props": "slimProp",
    "battery": "packMax",
    "sensor": "stareEoIr",
    "material": "carbonFiber",
}
isr = m0.interpret(model, "UavMissions::IsrUav", selection=ISR_MIX)
print("root:   ", isr.root)
print("battery:", isr.root.slots["battery"].id, " (a singleton omits the index)")
for motor in isr.root.slots["motors"]:
    print(f"  {motor.id}  mass={motor.slots['mass']} kg  maxThrust={motor.slots['maxThrust']} N")
sensor = isr.root.slots["sensor"]
print("sensor: ", sensor.id, f" ({sensor.slots['mass']} kg of stabilized optics)")
motor_count = len(isr.individuals("UavMissions::Motor"))
print("individuals:", len(isr.individuals()), " of which motors:", motor_count)

### A feature reads as a set of sequences (KerML Annex A)

Annex A of the KerML specification interprets a feature as a set of
*sequences* whose prefix is an individual of the featuring type:
`sequences("motors")` yields `(aircraft, motor_i)` pairs, and the
nested `sequences("motors.mass")` extends each pair by one more step.

What to look for: four 3-tuples, one per actual motor station, each
ending in that motor's own value.

In [ ]:
for seq in isr.sequences("motors")[:2]:
    print(seq)
print("...\n")
for owner, motor, mass in isr.sequences("motors.mass"):
    print(f"({owner.id}, {motor.id}, {mass})")

## Roll-ups weigh the individuals that exist; the M1 build-ups cannot

Look at how the catalog builds its mass ledger:
`airframe.motorCount * (motors.mass + props.mass)` -- a hand-encoded
scale factor standing in for a population, the same shortcut as
`4.0 * 0.06` in a toy model, just better dressed. Tutorial 7's trades
machinery evaluates it by treating `motors` as *one prototype* and
multiplying. Over a real population `motors.mass` is four values, and
multiplying a count by a list of masses is not an answer. The ratified
design decision: M1/M0 divergence **degrades to `None` and lands in
`Interpretation.gaps` -- it never raises**. The population still
builds, the holes are honest, and a strict caller simply asserts
`gaps == []`.

`rollup` is the M0 route: feature references resolve against the root
individual's slots, so `sum(motors.mass)` adds the four *real* motors
and `size(motors)` counts them -- no count is ever hand-encoded.

What to look for: the convention-built slots read `None`; every metric
the mission context itself declares lands in `gaps` with its reason
(the `None`s and four-value lists *inside* the reasons show the
cascade); and the honest sums underneath.

In [ ]:
print("baseMass slot:      ", isr.root.slots["baseMass"])
print("stationMinutes slot:", isr.root.slots["stationMinutes"])
print("\ngaps -- the mission's own metrics, each with its reason:")
for gap in isr.gaps:
    print("  ", gap)
motor_mass = isr.rollup("sum(motors.mass)")
print("\nsum(motors.mass):", motor_mass, "kg over", isr.rollup("size(motors)"), "motors")
print("sum(props.mass): ", round(isr.rollup("sum(props.mass)"), 3), "kg")

### The trap: two hand-encodings of one population, and they disagree

Every `MissionUAV` declares `motors : MotorChoice[4]` -- four
stations, hand-encoded. The `DartInterceptor` is a single-pusher
aircraft, and rather than change the multiplicity the model patches
the ledger with `airframe.motorCount = 1`. At M1 the patch works
perfectly: the trades machinery scales by `motorCount`, tutorial 7
scored the dart correctly, and nothing ever noticed that the model
*also* says four motors exist. Build the population and the
contradiction turns physical: the interpretation faithfully expands
`[4]` into four `SprintMotor` individuals on an aircraft that mounts
one, and the two mass ledgers split by half a kilogram.

What to look for: two confident, *different* masses for the same mix
-- the M1 number scaled by `motorCount`, the M0 number weighing the
declared individuals -- a 14% divergence with no exception anywhere,
just `gaps` entries showing the scalar convention refusing over
four-value lists. Which number is right? Here, the M1 one -- but only
the M0 population forces the question. The honest fix is a model fix
(`motors : MotorChoice[1]` for the dart family, or a `motorCount`
derived from `size(motors)`), and interpreting the architecture is how
the drift gets caught.

In [ ]:
from longeron.analysis.trades import TradeStudy

intercept = TradeStudy(model, "UavMissions::InterceptUav")
fastest = max(
    (a for a in intercept.all_architectures() if a.verified),
    key=lambda a: a.metrics["maxTargetSpeed"],
)
print("fastest feasible mix:", fastest.selection)

dart = m0.interpret(model, "UavMissions::InterceptUav", selection=fastest.selection)
stations = dart.root.slots["motors"]
print("declared stations:  ", [s.id.rsplit(".", 1)[-1] for s in stations])
print("motor type:", stations[0].type_name, end="   ")
print("airframe.motorCount:", dart.root.slots["airframe"].slots["motorCount"], "\n")

SPAR_MASS = (  # the wing spar, sized from loads in the selected material
    "TubeMass(radius = airframe.sparRadius, wall = max(minWallM,"
    " TubeWallForStress(momentNm = SparRootMoment(grossKg = airframe.designGrossKg,"
    " span = airframe.wingSpan), radius = airframe.sparRadius,"
    " yieldPa = material.yieldPa)), length = airframe.wingSpan,"
    " density = material.density)"
)
DART_LEDGER = (
    "airframe.mass + avionicsMass + battery.mass + "
    + SPAR_MASS
    + " + sum(motors.mass) + sum(props.mass) + seekerMass"
)
m1_mass = fastest.metrics["missionMass"]
m0_mass = dart.rollup(DART_LEDGER)
print(f"M1 mission mass (motorCount x per-unit):  {m1_mass:.3f} kg")
print(f"M0 mission mass (sum over individuals):   {m0_mass:.3f} kg")
print(f"three phantom motors and props:           {m0_mass - m1_mass:+.3f} kg\n")
for gap in dart.gaps[:3]:
    print("  ", gap)
print("   ...")

## Nominal takes the lower bound; random explores the range

The catalog pins its multiplicities at `[4]`, so a ten-line inline
fleet introduces the ranged case: `rotors : Rotor[2..6]` and
`spares : Rotor[0..*]`. Under **nominal** both take their lower bound
-- conservative, deterministic, and consistent with `instantiate()`;
that default is the other ratified decision. `strategy="random"` draws
population sizes uniformly within the bounds (an unbounded upper is
capped at `lower + 3`), samples unvalued enum and Boolean attributes
from their literal domains, and is fully seeded: equal seeds reproduce
equal populations, and `sample(n)` derives `n` fresh interpretations
from the parent seed. The next section takes exactly this machinery
back to the catalog.

What to look for: nominal always answers 2 rotors and 0 spares; the
seeded draws vary within `[2..6]` and `[0..3]`; and the rotors'
liveries carry sampled enum values.

In [ ]:
FIELD = """
package Fleet {
    enum def Livery { plain; racing; stealth; }
    part def Rotor {
        attribute mass : Real = 0.06;
        attribute livery : Livery;
    }
    part def FieldQuad {
        part rotors : Rotor[2..6];
        part spares : Rotor[0..*];
    }
}
"""
fleet = longeron.loads(FIELD)


def shape(it):
    return f"{len(it.root.slots['rotors'])} rotors, {len(it.root.slots['spares'])} spares"


print("nominal:", shape(m0.interpret(fleet, "Fleet::FieldQuad")))
drawn = m0.interpret(fleet, "Fleet::FieldQuad", strategy="random", seed=7)
print("seed 7: ", shape(drawn))
print("liveries:", [rotor.slots["livery"].name for rotor in drawn.root.slots["rotors"]])
for s in drawn.sample(3):
    print(f"  sample seed {s.seed}: {shape(s)}")
rerun = m0.interpret(fleet, "Fleet::FieldQuad", strategy="random", seed=7)
print("equal seeds reproduce equal populations:", rerun.to_dict() == drawn.to_dict())

## Monte-Carlo over the catalog: 64 aircraft nobody enumerated

`strategy="random"` on `IsrUav` draws **every** variation point -- and
it draws the motor and prop stations *per individual*: `motors#0` can
be a sprint motor while `motors#1` is an eco. Those heterogeneous
aircraft are populations the M1 convention cannot even score (there is
no single prototype to scale by `motorCount`), but the M0 ledger does
not care: each drawn aircraft is a population, and the roll-up weighs
whatever hangs on it. Sixty-four seeded draws (`sample(63)` plus the
parent) give a mass distribution over the whole catalog. The ledger is
the equipped hardware mass -- shell, avionics, pack, sized spar,
stations, sensor; the regression at the end of this tutorial proves it
equal to `missionMass` for wingtip-prop families (for the arm-built
quads it stops a few tens of grams short, at the rotor-arm tubes).

What to look for: sixty-four dots. The airframe families do order the
bands, but the *within*-family spread is wider than the spread between
family means -- pack and sensor draws move an aircraft by more than
choosing a different airframe does.

In [ ]:
from random import Random
from statistics import mean, stdev

import matplotlib.pyplot as plt

HARDWARE_MASS = (
    "airframe.mass + avionicsMass + battery.mass + "
    + SPAR_MASS
    + " + sum(motors.mass) + sum(props.mass) + sensor.mass"
)
parent = m0.interpret(model, "UavMissions::IsrUav", strategy="random", seed=2025)
fleet64 = [parent, *parent.sample(63)]
by_family: dict[str, list[float]] = {}
for it in fleet64:
    by_family.setdefault(it.selection["airframe"], []).append(it.rollup(HARDWARE_MASS))
masses = [mass for family in by_family.values() for mass in family]
print(
    f"64 seeded draws: {min(masses):.2f} .. {max(masses):.2f} kg,"
    f" mean {mean(masses):.2f}, sigma {stdev(masses):.2f}"
)

jitter = Random(0)
order = sorted(by_family, key=lambda f: mean(by_family[f]))
fig, ax = plt.subplots(figsize=(7.0, 3.0), layout="constrained")
for row, family in enumerate(order):
    xs = by_family[family]
    ys = [row + jitter.uniform(-0.18, 0.18) for _ in xs]
    ax.plot(xs, ys, "o", color="#2f6b8f", markersize=4.5, alpha=0.55, markeredgewidth=0)
    ax.plot(mean(xs), row, "|", color="#c2603e", markersize=18, markeredgewidth=2.2)
ax.set_yticks(range(len(order)), [f"{f} ({len(by_family[f])})" for f in order])
ax.set_xlabel("equipped hardware mass (kg)")
ax.set_title(
    "64 seeded draws over the ISR catalog: equipment choices outweigh the airframe",
    fontsize=10,
    loc="left",
    color="#2b2d31",
)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.grid(axis="x", color="#d9dbdf", linewidth=0.5)

That spread is a design input, not trivia. Every mix in this catalog
rides on shared infrastructure -- transport cases, launch rails, bench
chargers, spare-pack logistics -- and that infrastructure must cover
the *envelope*, 2.1 to 7.0 kg, not the 4.1-kg mean. The sigma of ~1.2
kg says most of that uncertainty is *configuration* uncertainty:
freeze the airframe and the mass has still only narrowed to a ~3-kg
band, because the pack (0.5-1.95 kg) and sensor (0.2-1.9 kg) draws
dominate. It also explains a number the catalog already carries: the
`vtolWing` band starting near 3.7 kg and running past 6.9 is why its
spar-sizing input `designGrossKg` is 6.0 -- the structure is sized for
the heavy corner of the catalog, not the nominal build. A seeded
distribution like this, over populations the M1 convention cannot
score, is the cheap first look at design margin -- one step before
anything as formal as tutorial 7's trade fronts.

## Traces are interpretations

pymbe, the reference implementation for these population semantics,
could describe individuals but never execute anything. longeron
already executes state machines, and a recorded execution *is* an
interpretation of the behavior. `from_timeline` turns every contiguous
state activation of tutorial 4's `Drone::FlightStates` into an
**occurrence individual** (`qname@k`) with `start`/`end`/`duration`
slots, owned by a root that spans the whole recording.

What to look for: the flight reads as a population of five occurrences
in activation order, and the second visit to `idle` gets a fresh
identity, `@1`.

In [ ]:
from longeron.replay import record_timeline

drone = longeron.load("../examples/drone.sysml")
interp = longeron.Interpreter(drone)
flight = record_timeline(
    interp,
    "Drone::FlightStates",
    [1.5, "launch", 2.0, "airborne", 10.0, "low_battery", 1.0, "touchdown"],
)
trace = m0.from_timeline(flight, source="Drone::FlightStates")
print("strategy:", trace.strategy, " recording span:", trace.root.slots["duration"], "s\n")
for occ in trace.root.slots["occurrences"]:
    start, end = occ.slots["start"], occ.slots["end"]
    print(f"  {occ.id:36s} {start:5.1f} -> {end:5.1f}  ({occ.slots['duration']:4.1f} s)")
print("\nidle re-entries:", [ind.id for ind in trace.individuals("Drone::FlightStates::idle")])
print("sum(occurrences.duration):", trace.rollup("sum(occurrences.duration)"))

### One representation, two kinds of individual

A catalog motor and a recorded occurrence are the same class, queried
by the same machinery -- only their slots differ, a datasheet versus a
lifetime. That is why `rollup` and `sequences` work unchanged over
executions: `sum(occurrences.duration)` above is the same operation as
`sum(motors.mass)`.

In [ ]:
static = isr.root.slots["motors"][2]
occurrence = trace.individuals("Drone::FlightStates::flying")[0]
for individual in (static, occurrence):
    print(f"{individual!r}\n    slots: {dict(individual.slots)}")
print("same class:", type(static) is type(occurrence) is m0.Individual)

## A trades `Architecture` is a partial interpretation

Tutorial 7 enumerated 648 ISR mixes by pinning every variation point
and scoring the result through the interpreter. That pinning *is* a
partial M0 interpretation: variant selection fixed, population
nominal. `from_architecture` makes it literal -- and closes this
tutorial's loop: run the study, take the winner, and it is `ISR_MIX`,
the very selection interpreted in the first cell.

What to look for: the winner's selection equals `ISR_MIX`, and the
interpretation built from the architecture equals -- `to_dict()` for
`to_dict()` -- the one built by hand. An architecture is not *like* an
interpretation; it denotes the same population.

In [ ]:
study = TradeStudy(model, "UavMissions::IsrUav")
architectures = study.all_architectures()
feasible = [a for a in architectures if a.verified]
print(len(architectures), "mixes,", len(feasible), "feasible")

winner = max(feasible, key=lambda a: a.metrics["stationMinutes"])
print("longest-station mix:", winner.selection)
print("tutorial 7's winner is the mix from the first cell:", winner.selection == ISR_MIX)

isr_m0 = m0.from_architecture(study, winner)
print("same population, individual for individual:", isr_m0.to_dict() == isr.to_dict())

### The regression: individuals reproduce the trades metrics

Rewrite each ISR metric over the actual population -- per-unit values
recovered as `sum(x) / size(x)`, so no count is hand-encoded anywhere
-- and it must equal the number the trades machinery computed at M1:
same model, two population semantics, one answer, down to the 147.386
minutes on station. `tests/test_m0.py` pins the same discipline across
all 54 mixes of the quad-copter catalog, so the two semantics cannot
drift apart unnoticed.

What to look for: four exact matches, including the full physics chain
-- momentum-theory hover reserve, wing-borne loiter power, the sized
carbon spar -- rebuilt from individuals.

In [ ]:
import math

DISK_AREA = (
    "airframe.diskAreaFactor * 3.141592653589793"
    " * pow(sum(props.diameter) / size(props), 2.0) / 4.0"
)
HOVER = f"HoverPower(massKg = {HARDWARE_MASS}, diskArea = {DISK_AREA})"
LOITER = (
    f"CruisePower(massKg = {HARDWARE_MASS}, speed = airframe.loiterSpeed,"
    " dragArea = airframe.dragArea, span = airframe.wingSpan,"
    " wingArea = airframe.wingArea,"
    " spanEff = airframe.oswald * airframe.tipPropBonus,"
    " propEff = sum(props.cruiseEff) / size(props)"
    " * sum(motors.efficiency) / size(motors))"
)
ROLLUPS = {
    "missionMass": HARDWARE_MASS,
    "missionCost": (
        "airframe.cost + avionicsCost + battery.cost"
        f" + ({SPAR_MASS}) * material.costPerKg"
        " + sum(motors.cost) + sum(props.cost) + sensor.cost"
    ),
    "maxThrust": "sum(motors.maxThrust)",
    "stationMinutes": (
        f"(usableEnergyJ - ({HOVER}) * airframe.hoverOpsSec) / (({LOITER}) + sensor.powerW) / 60.0"
    ),
}
print(f"{'metric':16s}{'trades (M1)':>14s}{'roll-up (M0)':>14s}")
for metric, expr in ROLLUPS.items():
    value = isr_m0.rollup(expr)
    assert math.isclose(value, winner.metrics[metric], rel_tol=1e-12)
    print(f"{metric:16s}{winner.metrics[metric]:14.3f}{value:14.3f}")

## The JSON shape stays out of the standard API

`to_dict()` projects an interpretation -- ids, selection, gaps, the
full slot tree -- into plain JSON-able data. It is a deliberate
longeron *extension*: the OMG Systems Modeling API has no M0
representation, so `to_api_json` (tutorial 2) never emits it, keeping
the standard record stream clean for ecosystem consumers -- the third
ratified decision; if interpretations are ever served over HTTP they
enter through an extension namespace instead.

In [ ]:
import json

payload = isr.to_dict()
print("keys:", list(payload))
print(json.dumps(payload["root"]["motors"][0], indent=2))
print("JSON round-trip:", json.loads(json.dumps(payload)) == payload)

The M0 story in one line: **one `Individual` representation, from a
real catalog's part trees through seeded Monte-Carlo draws to recorded
occurrences and trade-study architectures**, with `rollup` and
`sequences` as the single query surface and `gaps` as the honesty
channel. The design rationale, the three ratified decisions this
tutorial demonstrated, and what comes next (an RDF projection of
individuals, heterogeneous per-index selections feeding back into the
trade study) live in `docs/design/m0-interpretations.md`.